In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, RepeatVector, Dense
from sklearn.preprocessing import StandardScaler

from pathlib import Path

dataset_root = Path("LP IV Updated Datasets")
if not dataset_root.is_dir():
    dataset_root = Path("../LP IV Updated Datasets")
print("Dataset folder:", dataset_root.resolve())


In [ ]:
# Load readings in time order
df = pd.read_csv(dataset_root / "AutoEncoder/Energy Consumption/energy consumption.csv")
print("Loaded data shape:", df.shape)
print("First three rows:")
print(df.head(3))
print("Label counts:")
print(df.Anomaly_Label.value_counts().sort_index())

df["Timestamp"] = pd.to_datetime(df.Timestamp, dayfirst=True)
df = df.sort_values("Timestamp")
features = df.drop(columns=["Timestamp", "Anomaly_Label"]).columns

# Fill missing measurements using earlier values, or zero for initial gaps
print("Missing feature values before filling:", df[features].isna().sum().sum())
df[features] = df[features].ffill().fillna(0)
print("Missing feature values after filling:", df[features].isna().sum().sum())

# Use earlier readings for training and later readings for testing
split = int(len(df) * 0.8)
train = df.iloc[:split]
test = df.iloc[split:]

# Learn scaling from normal training readings
normal_train = train[train.Anomaly_Label == 0]
scaler = StandardScaler()
scaler.fit(normal_train[features])
train_values = scaler.transform(train[features])
test_values = scaler.transform(test[features])
train_labels = train.Anomaly_Label.to_numpy()
window_size = 24

# Build training windows containing only normal readings
x_train = []
for end in range(window_size, len(train_values) + 1):
    start = end - window_size
    if train_labels[start:end].sum() == 0:
        x_train.append(train_values[start:end])
x_train = np.array(x_train)

# Build all test windows
x_test = []
for end in range(window_size, len(test_values) + 1):
    start = end - window_size
    x_test.append(test_values[start:end])
x_test = np.array(x_test)

print("Training input shape:", x_train.shape)
print("Test input shape:", x_test.shape)
print("Training value range:", x_train.min(), "to", x_train.max())
print("Test value range:", x_test.min(), "to", x_test.max())
print("Selected features:", list(features))
print("Window size:", window_size)
print("First training window: first three readings and five features")
print(x_train[0, :3, :5])
print("Normal training readings:", len(normal_train))
print("Training/test time ranges:", train.Timestamp.min(), "to", train.Timestamp.max(), "/", test.Timestamp.min(), "to", test.Timestamp.max())


In [ ]:
# Create LSTM Autoencoder
model = Sequential([
    Input(shape=(window_size, len(features))),
    LSTM(16),
    RepeatVector(window_size),
    LSTM(16, return_sequences=True),
    Dense(len(features))
])
model.compile(optimizer="adam", loss="mse")
model.summary()


In [ ]:
# Train using normal inputs as their own reconstruction targets
history = model.fit(x_train, x_train, epochs=10, batch_size=128)
print("Last three training epochs:")
print(pd.DataFrame(history.history).tail(3))


In [ ]:
# Reconstruction errors for normal training inputs
train_prediction = model.predict(x_train)
train_error = np.mean((x_train - train_prediction) ** 2, axis=(1, 2))
print("First five training errors:", train_error[:5])
print("Training error min / mean / max:", train_error.min(), train_error.mean(), train_error.max())
threshold = np.percentile(train_error, 95)
print("Anomaly threshold (95th percentile):", threshold)


In [ ]:
# Reconstruction errors for test inputs
test_prediction = model.predict(x_test)
test_error = np.mean((x_test - test_prediction) ** 2, axis=(1, 2))
anomalies = test_error > threshold
print("First ten test errors:", test_error[:10])
print("Test error min / mean / max:", test_error.min(), test_error.mean(), test_error.max())
print("First ten detected flags:", anomalies[:10])
print("Anomalies:", anomalies.sum())
print("First ten test windows:")
print(pd.DataFrame({
    "Error": test_error[:10],
    "Detected anomaly": anomalies[:10]
}))
